# Block Prediction Analysis

This notebook analyzes how the T5 model predicts block-level locations and investigates:
1. Whether block names match village names (model copying strategy)
2. Whether block names match district names
3. Whether blocks are actually mentioned in the incident descriptions
4. Whether blocks were coded from the text or added during geocoding

The goal is to understand why block-level predictions have low accuracy (38.85% precision, 41.44% F1) and determine whether to keep them in the geocoding pipeline.

**Note:** This notebook is designed to run in Google Colab and loads results from the location_extraction.ipynb analysis stored in Google Drive.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("Libraries imported successfully")

## Mount Google Drive

Mount Google Drive to access the saved prediction results.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted successfully")

## Load Prediction Data

Load the predictions from the T5 location extraction model stored in Google Drive.

In [ ]:
# Set the path to your Google Drive results folder
# Adjust this path based on where your location_extraction.ipynb saves results
drive_path = "/content/drive/MyDrive/satp-results/"  # Update this path as needed

# Load the predictions CSV
predictions_df = pd.read_csv(f"{drive_path}location_extraction_test_predictions.csv")

print(f"Total predictions: {len(predictions_df)}")
print(f"\nColumns: {predictions_df.columns.tolist()}")
print(f"\nFirst few rows:")
predictions_df.head()

## Analysis 1: Block Prediction Strategies

Investigate how the model generates block predictions and what strategies it uses.

In [ ]:
# Filter to cases where block was predicted
block_predictions = predictions_df[predictions_df['pred_block'].notna()].copy()

print(f"Total predictions with blocks: {len(block_predictions)}")
print(f"Correct block predictions: {block_predictions['block_match'].sum()}")
print(f"Block accuracy: {block_predictions['block_match'].mean()*100:.2f}%\n")

# HYPOTHESIS 1: Block name matches village name
block_predictions['block_equals_village'] = block_predictions.apply(
    lambda row: (row['pred_block'] or '').lower() == (row['pred_village'] or '').lower() 
    if pd.notna(row['pred_block']) and pd.notna(row['pred_village']) else False,
    axis=1
)

# HYPOTHESIS 2: Block name matches district name
block_predictions['block_equals_district'] = block_predictions.apply(
    lambda row: (row['pred_block'] or '').lower() == (row['pred_district'] or '').lower()
    if pd.notna(row['pred_block']) and pd.notna(row['pred_district']) else False,
    axis=1
)

# HYPOTHESIS 3: Block name is substring of input text (actually mentioned)
block_predictions['block_in_input'] = block_predictions.apply(
    lambda row: (row['pred_block'] or '').lower() in (row['input_text'] or '').lower()
    if pd.notna(row['pred_block']) and pd.notna(row['input_text']) else False,
    axis=1
)

print("="*80)
print("BLOCK PREDICTION STRATEGIES")
print("="*80)

# Strategy 1: Block = Village
strategy1 = block_predictions[block_predictions['block_equals_village']]
print(f"\nStrategy 1: Block name = Village name")
print(f"  Frequency: {len(strategy1)}/{len(block_predictions)} ({len(strategy1)/len(block_predictions)*100:.1f}%)")
if len(strategy1) > 0:
    print(f"  Accuracy when used: {strategy1['block_match'].mean()*100:.1f}%")
    print(f"  Example:")
    sample = strategy1.iloc[0]
    print(f"    Predicted block: {sample['pred_block']}")
    print(f"    Predicted village: {sample['pred_village']}")
    print(f"    Ground truth block: {sample['gt_block']}")
    print(f"    Correct? {sample['block_match']}")

# Strategy 2: Block = District
strategy2 = block_predictions[block_predictions['block_equals_district'] & ~block_predictions['block_equals_village']]
print(f"\nStrategy 2: Block name = District name (when not also village)")
print(f"  Frequency: {len(strategy2)}/{len(block_predictions)} ({len(strategy2)/len(block_predictions)*100:.1f}%)")
if len(strategy2) > 0:
    print(f"  Accuracy when used: {strategy2['block_match'].mean()*100:.1f}%")

# Strategy 3: Block actually mentioned in text
strategy3 = block_predictions[block_predictions['block_in_input']]
print(f"\nStrategy 3: Block name appears in input text")
print(f"  Frequency: {len(strategy3)}/{len(block_predictions)} ({len(strategy3)/len(block_predictions)*100:.1f}%)")
if len(strategy3) > 0:
    print(f"  Accuracy when used: {strategy3['block_match'].mean()*100:.1f}%")
    correct_and_mentioned = strategy3[strategy3['block_match']]
    if len(correct_and_mentioned) > 0:
        print(f"  Example where mentioned and correct:")
        sample = correct_and_mentioned.iloc[0]
        print(f"    Input snippet: ...{sample['input_text'][200:400]}...")
        print(f"    Predicted block: {sample['pred_block']}")
        print(f"    Ground truth block: {sample['gt_block']}")

# Strategy 4: None of the above (pure hallucination/learned patterns)
strategy4 = block_predictions[
    ~block_predictions['block_equals_village'] & 
    ~block_predictions['block_equals_district'] & 
    ~block_predictions['block_in_input']
]
print(f"\nStrategy 4: None of the above (learned geographic patterns or hallucination)")
print(f"  Frequency: {len(strategy4)}/{len(block_predictions)} ({len(strategy4)/len(block_predictions)*100:.1f}%)")
if len(strategy4) > 0:
    print(f"  Accuracy when used: {strategy4['block_match'].mean()*100:.1f}%")

print("\n" + "="*80)
print("BREAKDOWN OF CORRECT BLOCK PREDICTIONS")
print("="*80)

correct_blocks = block_predictions[block_predictions['block_match'] == True]
print(f"\nOf the {len(correct_blocks)} correct block predictions:")
print(f"  {(correct_blocks['block_equals_village'].sum())} via Strategy 1 (block=village)")
print(f"  {(correct_blocks['block_equals_district'].sum())} via Strategy 2 (block=district)")  
print(f"  {(correct_blocks['block_in_input'].sum())} via Strategy 3 (mentioned in text)")
other_correct = len(correct_blocks) - correct_blocks[['block_equals_village', 'block_equals_district', 'block_in_input']].any(axis=1).sum()
print(f"  {other_correct} via Strategy 4 (learned patterns)")

## Analysis 2: Ground Truth Block-Village Relationship

Check how often block names match village names in the ground truth data. This tells us if the model learned a valid heuristic.

In [ ]:
print("GROUND TRUTH ANALYSIS: Block-Village Name Matching")
print("="*80)

# Load original location data
url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/location_info.csv"
location_data = pd.read_csv(url, dtype=str)

print(f"Total records in location data: {len(location_data)}")

# Cases with both block and village
both_present = location_data[location_data['block'].notna() & location_data['village_name'].notna()].copy()
print(f"Records with both block and village: {len(both_present)}")

# Check if block name matches village name
both_present['names_match'] = both_present['block'].str.lower() == both_present['village_name'].str.lower()
matches = both_present[both_present['names_match']]

print(f"\nRecords where block name = village name: {len(matches)} ({len(matches)/len(both_present)*100:.1f}%)")
print(f"\n⚠️  This means 'block=village' is a valid strategy {len(matches)/len(both_present)*100:.1f}% of the time")
print(f"    The model learned this pattern from training data!")

# Show examples
print(f"\nExamples where block = village:")
for idx, row in matches.head(5).iterrows():
    print(f"  - Block: {row['block']}, Village: {row['village_name']}, District: {row['district']}, State: {row['state']}")

print(f"\nExamples where block ≠ village:")
not_matches = both_present[~both_present['names_match']]
for idx, row in not_matches.head(5).iterrows():
    print(f"  - Block: {row['block']}, Village: {row['village_name']}, District: {row['district']}, State: {row['state']}")

## Analysis 3: Are Blocks Mentioned in Incident Descriptions?

Investigate whether blocks in the ground truth data were coded from the incident descriptions or added later (e.g., during geocoding).

In [ ]:
print("GROUND TRUTH ANALYSIS: Block Mentions in Incident Descriptions")
print("="*80)

# Load incident summaries
summary_url = "https://raw.githubusercontent.com/eteitelbaum/code-satp/refs/heads/main/data/satp_clean.csv"
satp_data = pd.read_csv(summary_url, dtype=str)

# Merge location data with incident summaries
merged = location_data.merge(
    satp_data[['shrid', 'incident_summary']], 
    on='shrid', 
    how='left'
)

print(f"Total records with incident summaries: {merged['incident_summary'].notna().sum()}")

# Filter to cases with blocks
with_blocks = merged[merged['block'].notna() & merged['incident_summary'].notna()].copy()
print(f"Records with both block and incident summary: {len(with_blocks)}")

# Check if block name appears in incident summary
with_blocks['block_in_summary'] = with_blocks.apply(
    lambda row: row['block'].lower() in row['incident_summary'].lower() 
    if pd.notna(row['block']) and pd.notna(row['incident_summary']) else False,
    axis=1
)

mentioned = with_blocks[with_blocks['block_in_summary']]
not_mentioned = with_blocks[~with_blocks['block_in_summary']]

print(f"\nBlocks mentioned in incident summary: {len(mentioned)} ({len(mentioned)/len(with_blocks)*100:.1f}%)")
print(f"Blocks NOT mentioned in incident summary: {len(not_mentioned)} ({len(not_mentioned)/len(with_blocks)*100:.1f}%)")

print(f"\n{'='*80}")
print(f"🔍 INTERPRETATION:")
print(f"{'='*80}")

if len(mentioned) / len(with_blocks) < 0.3:
    print(f"\n⚠️  Only {len(mentioned)/len(with_blocks)*100:.1f}% of blocks appear in incident descriptions!")
    print(f"\n   This suggests blocks were likely:")
    print(f"   1. Added during geocoding/post-processing")
    print(f"   2. Inferred from village names using administrative databases")
    print(f"   3. Not reliably derivable from the incident text alone")
    print(f"\n   RECOMMENDATION: Don't expect the model to extract what's not in the text!")
    print(f"   Consider removing blocks from the extraction schema.")
elif len(mentioned) / len(with_blocks) > 0.7:
    print(f"\n✓ {len(mentioned)/len(with_blocks)*100:.1f}% of blocks appear in incident descriptions.")
    print(f"\n   This suggests blocks were coded from the text.")
    print(f"   The model should be able to learn this with better training.")
else:
    print(f"\n⚠️  {len(mentioned)/len(with_blocks)*100:.1f}% of blocks appear in incident descriptions.")
    print(f"\n   Mixed results - some blocks are mentioned, many are not.")
    print(f"   Consider a hybrid approach or confidence filtering.")

# Show examples
print(f"\n{'='*80}")
print(f"EXAMPLES WHERE BLOCK IS MENTIONED:")
print(f"{'='*80}")
for idx, row in mentioned.head(3).iterrows():
    print(f"\nBlock: {row['block']}")
    print(f"District: {row['district']}, State: {row['state']}")
    print(f"Incident: {row['incident_summary'][:200]}...")

print(f"\n{'='*80}")
print(f"EXAMPLES WHERE BLOCK IS NOT MENTIONED:")
print(f"{'='*80}")
for idx, row in not_mentioned.head(3).iterrows():
    print(f"\nBlock: {row['block']} (not found in text)")
    print(f"Village: {row['village_name']}, District: {row['district']}, State: {row['state']}")
    print(f"Incident: {row['incident_summary'][:200]}...")

## Visualization: Strategy Distribution

Visualize which strategies the model uses and their success rates.

In [ ]:
# Create strategy labels for each prediction
def assign_strategy(row):
    if row['block_equals_village']:
        return 'Block=Village'
    elif row['block_equals_district']:
        return 'Block=District'
    elif row['block_in_input']:
        return 'Mentioned in Text'
    else:
        return 'Learned Pattern'

block_predictions['strategy'] = block_predictions.apply(assign_strategy, axis=1)

# Create summary dataframe
strategy_summary = block_predictions.groupby('strategy').agg({
    'block_match': ['count', 'sum', 'mean']
}).reset_index()
strategy_summary.columns = ['Strategy', 'Total', 'Correct', 'Accuracy']
strategy_summary['Accuracy'] = strategy_summary['Accuracy'] * 100

print("\nStrategy Summary:")
print(strategy_summary)

# Plot 1: Strategy frequency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequency
axes[0].bar(strategy_summary['Strategy'], strategy_summary['Total'], color='steelblue', alpha=0.7)
axes[0].set_xlabel('Strategy', fontsize=12)
axes[0].set_ylabel('Number of Predictions', fontsize=12)
axes[0].set_title('Block Prediction Strategy Frequency', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
for i, v in enumerate(strategy_summary['Total']):
    axes[0].text(i, v + 5, str(v), ha='center', fontsize=10)

# Accuracy
colors = ['green' if acc > 50 else 'orange' if acc > 30 else 'red' for acc in strategy_summary['Accuracy']]
axes[1].bar(strategy_summary['Strategy'], strategy_summary['Accuracy'], color=colors, alpha=0.7)
axes[1].set_xlabel('Strategy', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Block Prediction Accuracy by Strategy', fontsize=14, fontweight='bold')
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% threshold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()
for i, v in enumerate(strategy_summary['Accuracy']):
    axes[1].text(i, v + 2, f"{v:.1f}%", ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## Summary and Recommendations

Based on the analyses above, generate final recommendations.

In [ ]:
print("="*80)
print("FINAL SUMMARY AND RECOMMENDATIONS")
print("="*80)

# Calculate key metrics
pct_block_in_text = len(mentioned) / len(with_blocks) * 100
pct_block_eq_village_gt = len(matches) / len(both_present) * 100
pct_block_eq_village_pred = len(strategy1) / len(block_predictions) * 100
block_precision = block_predictions['block_match'].mean() * 100

print(f"\n📊 KEY FINDINGS:")
print(f"  - Block precision: {block_precision:.1f}%")
print(f"  - Blocks mentioned in incident text: {pct_block_in_text:.1f}%")
print(f"  - Blocks match village names in ground truth: {pct_block_eq_village_gt:.1f}%")
print(f"  - Model uses 'block=village' strategy: {pct_block_eq_village_pred:.1f}%")

print(f"\n💡 RECOMMENDATIONS:\n")

if pct_block_in_text < 30 and block_precision < 50:
    print("  🔴 STRONG RECOMMENDATION: Remove blocks from the extraction pipeline")
    print("\n  Reasons:")
    print(f"    1. Only {pct_block_in_text:.1f}% of blocks appear in incident descriptions")
    print(f"    2. Model precision is only {block_precision:.1f}% - more wrong than right")
    print(f"    3. Including wrong blocks will hurt geocoding accuracy")
    print("\n  Alternative approach:")
    print("    - Merge 'block' and 'other_location_info' into 'other_location' field")
    print("    - Let model extract sub-district info without forcing administrative categories")
    print("    - Focus on state (93% F1), district (92% F1), village (78% F1)")
elif pct_block_in_text > 60 and block_precision < 50:
    print("  🟡 RECOMMENDATION: Improve block extraction through better training")
    print("\n  Reasons:")
    print(f"    1. {pct_block_in_text:.1f}% of blocks ARE in the text - extractable in principle")
    print(f"    2. Current precision ({block_precision:.1f}%) suggests training issues")
    print("\n  Potential improvements:")
    print("    - More training data with block annotations")
    print("    - Add 'block' as a specific extraction target in prompts")
    print("    - Use confidence filtering to remove uncertain predictions")
else:
    print("  🟡 RECOMMENDATION: Use confidence-based filtering for blocks")
    print("\n  Reasons:")
    print(f"    1. Blocks sometimes mentioned ({pct_block_in_text:.1f}%), sometimes not")
    print(f"    2. Model has some signal but low precision ({block_precision:.1f}%)")
    print("\n  Hybrid approach:")
    print("    - Keep block extraction but filter low-confidence predictions")
    print("    - Use sequence log-probability or beam search diversity")
    print("    - Only include blocks with high confidence in geocoding queries")

print(f"\n  ✅ What's working well:")
print(f"    - State extraction: 93% F1")
print(f"    - District extraction: 92% F1")
print(f"    - Village extraction: 78% F1")
print(f"    - These three levels provide strong geographic coverage")

print(f"\n  🔬 Next steps:")
print(f"    1. A/B test geocoding with vs without blocks")
print(f"    2. Measure distance error for both approaches")
print(f"    3. If blocks hurt accuracy, retrain model with 'other_location' schema")
print(f"\n{'='*80}")